# 모델 정교화 — 잔차 분석 + 성별 분리 발견

## 실험 목표
RMSE 0.30에서의 병목 원인을 분석하고, 새로운 개선 방향을 탐색해봄

## 실험 흐름
1. **Identity/Sqrt 결합** — 간소화된 2모델 앙상블 (RMSE 0.299)
2. **Round 적용** — 정수 타겟 특성 반영 (RMSE 0.221)
3. **잔차 구조 분석** — 곱셈 구조 발견 (Duration×BPM×Temp)
4. **대안 모델 탐색** — Degree 4 Ridge, MLP, LightGBM+Keyser 공식
5. **성별 분리 모델** — M/F 별도 학습 (RMSE 0.157, 핵심 돌파)

## 성능 요약
| 단계 | RMSE | 핵심 |
|------|------|------|
| identity/sqrt 결합 | 0.299 | 경량화 |
| + round | 0.221 | 정수 타겟 |
| + 잔차 분석 | - | 구조 이해 |
| + 성별 분리 | **0.157** | M/F 분리 |

---
## 1. 라이브러리 및 데이터 준비

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submission.csv')

---
## 2. 피처 엔지니어링 + 전처리 (03 파일과 동일)

In [ ]:
def create_features_safe(df):
    df = df.copy()
    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6
    df['Duration_bin'] = pd.cut(df['Exercise_Duration'], bins=[-np.inf,10,20,30,np.inf], labels=[0,1,2,3]).astype(int)
    df['Duration_x_BPM'] = df['Exercise_Duration'] * df['BPM']
    df['Duration_x_TempDiff'] = df['Exercise_Duration'] * df['Temp_diff']
    df['BPM_x_TempDiff'] = df['BPM'] * df['Temp_diff']
    df['Intensity'] = df['Duration_x_BPM']
    df['Effort'] = df['Weight(lb)'] * df['Intensity']
    df['Duration_sq'] = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq'] = df['Temp_diff'] ** 2
    df['Dur_BPM_TempDiff'] = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']
    df['BPM_per_Duration'] = df['BPM'] / (df['Exercise_Duration'] + 1)
    df['TempDiff_per_Duration'] = df['Temp_diff'] / (df['Exercise_Duration'] + 1)
    h2 = (df['Height_Total_Inches'] ** 2).replace(0, np.nan)
    df['BMI'] = 703 * df['Weight(lb)'] / h2
    df['BMI'] = df['BMI'].fillna(df['BMI'].median())
    df['Weight_x_Duration'] = df['Weight(lb)'] * df['Exercise_Duration']
    df['Log_BPM'] = np.log1p(df['BPM'])
    df['Log_Duration'] = np.log1p(df['Exercise_Duration'])
    df['Log_Weight_BPM_Dur'] = np.log1p(df['Weight(lb)'] * df['BPM'] * df['Exercise_Duration'])
    return df

# 전처리
train_x_raw = train.drop(['ID', 'Calories_Burned'], axis=1).copy()
train_y = train['Calories_Burned'].astype(float).copy()
test_x_raw = test.drop(['ID'], axis=1).copy()

before_cols = list(train_x_raw.columns)
train_x_feat = create_features_safe(train_x_raw)
test_x_feat = create_features_safe(test_x_raw)
generated_cols = [c for c in train_x_feat.columns if c not in before_cols]

# OneHot
cat_cols = [c for c in ['Gender', 'Weight_Status'] if c in train_x_feat.columns]
if cat_cols:
    enc = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    enc.fit(train_x_feat[cat_cols])
    tr_cat = enc.transform(train_x_feat[cat_cols])
    te_cat = enc.transform(test_x_feat[cat_cols])
    cat_names = enc.get_feature_names_out(cat_cols)
    train_x_feat = train_x_feat.drop(columns=cat_cols)
    test_x_feat = test_x_feat.drop(columns=cat_cols)
    train_x_feat[cat_names] = tr_cat
    test_x_feat[cat_names] = te_cat

# Pruning: base + TOP5
TOP5_FIXED = ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']
base_feats = [c for c in train_x_feat.columns if c not in generated_cols]
keep_cols = base_feats + TOP5_FIXED

train_red = train_x_feat[keep_cols].copy()
test_red = test_x_feat[keep_cols].copy()

y = train_y.values.astype(float)
print(f"Reduced features: {train_red.shape[1]}")

---
## 3. 실험 — Identity + Sqrt Ridge 결합

스태킹을 간소화: 8개 베이스 → 2개만 사용
- **identity Ridge**: 원단위 y로 학습 (메인 모델)
- **sqrt Ridge**: sqrt(y)로 학습 → 제곱 복원 (보조 모델)
- 최적 가중치: identity ~0.985, sqrt ~0.015 (거의 identity가 메인)

In [ ]:
DEGREE = 3
ALPHA_ID = 0.05   # identity Ridge 규제 (pruning 후 약하게 → 표현력 향상)
ALPHA_SQ = 0.009  # sqrt Ridge 규제
W_ID = 0.9851676708561896  # scipy 최적화로 찾은 가중치

y_sqrt = np.sqrt(np.clip(y, 0, None))

# OOF + Test 예측
id_oof = np.zeros(len(train_red)); id_test = np.zeros(len(test_red))
sq_oof = np.zeros(len(train_red)); sq_test = np.zeros(len(test_red))

for tr_idx, va_idx in kf.split(train_red):
    # Poly + Scaler: fold 내부에서만 fit (누수 방지)
    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    sc = StandardScaler()
    X_tr = sc.fit_transform(poly.fit_transform(train_red.iloc[tr_idx]))
    X_va = sc.transform(poly.transform(train_red.iloc[va_idx]))
    X_te = sc.transform(poly.transform(test_red))
    
    # identity 모델
    m_id = Ridge(alpha=ALPHA_ID, random_state=RANDOM_STATE)
    m_id.fit(X_tr, y[tr_idx])
    id_oof[va_idx] = m_id.predict(X_va)
    id_test += m_id.predict(X_te) / N_SPLITS
    
    # sqrt 모델 (sqrt 스케일로 학습 → 원단위 복원)
    m_sq = Ridge(alpha=ALPHA_SQ, random_state=RANDOM_STATE)
    m_sq.fit(X_tr, y_sqrt[tr_idx])
    sq_oof[va_idx] = np.clip(m_sq.predict(X_va), 0, None) ** 2  # 복원
    sq_test += (np.clip(m_sq.predict(X_te), 0, None) ** 2) / N_SPLITS

# 클리핑
id_oof = np.clip(id_oof, 0, None); sq_oof = np.clip(sq_oof, 0, None)
id_test = np.clip(id_test, 0, None); sq_test = np.clip(sq_test, 0, None)

# 가중 결합
final_oof = np.clip(W_ID * id_oof + (1 - W_ID) * sq_oof, 0, None)
final_test_raw = np.clip(W_ID * id_test + (1 - W_ID) * sq_test, 0, None)

print(f"결합 OOF RMSE: {rmse(y, final_oof):.6f}")

---
## 4. 실험 — 정수 반올림 (Round) 적용

**핵심 인사이트**: 타겟(Calories_Burned)이 정수값
→ 예측을 정수로 반올림하면 RMSE가 직접적으로 개선됨

추가로 라운딩 경계(0.5±ε)에서 보조 예측(sqrt)으로 tie-break하는 규칙 기반 보정도 시도

In [ ]:
# 단순 정수 반올림
final_oof_round = np.rint(np.clip(final_oof, 0, None))
final_test_round = np.rint(np.clip(final_test_raw, 0, None))

print(f"Round 전 RMSE: {rmse(y, final_oof):.6f}")
print(f"Round 후 RMSE: {rmse(y, final_oof_round):.6f}")

# 라운딩 경계 tie-break 보정 (optional)
eps = 0.004  # 0.5 ± eps 범위에서 보조 예측 참고
frac = final_oof - np.floor(final_oof)  # 소수 부분
boundary_mask = np.abs(frac - 0.5) < eps  # 0.5 근처인 샘플

# 경계 근처에서는 sqrt 예측 기준으로 반올림
adjusted = final_oof_round.copy()
adjusted[boundary_mask] = np.rint(np.clip(sq_oof[boundary_mask], 0, None))

print(f"Tie-break 보정 후 RMSE: {rmse(y, adjusted):.6f}")
print(f"경계 근처 샘플 수: {boundary_mask.sum()} / {len(y)}")

---
## 5. 잔차 구조 분석 — 곱셈 구조 발견

예측 잔차를 분석하여 데이터의 근본 구조를 파악
→ **Calories ≈ a × Duration × BPM + b × Duration × BPM × Temp_diff + ...** 형태의 곱셈 구조 발견

이 분석이 성별 분리 모델의 이론적 근거가 됨

In [ ]:
from numpy.linalg import lstsq

# 체온 편차 (정상체온 기준)
train['Temp_diff'] = train['Body_Temperature(F)'] - 98.6

# 1단계: Duration × BPM 만으로 회귀
X1 = np.column_stack([train['Exercise_Duration'] * train['BPM'], np.ones(len(train))])
coeffs1, _, _, _ = lstsq(X1, train['Calories_Burned'].values, rcond=None)
pred1 = X1 @ coeffs1
res1 = train['Calories_Burned'].values - pred1
print(f"1단계 (Duration×BPM): RMSE={rmse(train['Calories_Burned'].values, pred1):.4f}")

# 잔차 vs 각 변수 상관
print("\n잔차 상관분석:")
for col in ['Temp_diff', 'Age', 'Weight(lb)', 'Exercise_Duration', 'BPM']:
    print(f"  {col}: {np.corrcoef(res1, train[col].values)[0,1]:.4f}")

In [ ]:
# 2단계: 곱셈 구조 확장 (Duration×BPM×Temp_diff 등)
X_full = np.column_stack([
    train['Exercise_Duration'] * train['BPM'],                              # 기본 강도
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'],        # 강도 × 체온
    train['Exercise_Duration'] * train['BPM'] * train['Age'],             # 강도 × 나이
    train['Exercise_Duration'] * train['BPM'] * train['Temp_diff'] * train['Age'],  # 4변수 곱
    train['Age'],                                                          # 나이 단독
    train['Exercise_Duration'] * train['Weight(lb)'],                     # 시간 × 체중
    train['Weight(lb)'],                                                  # 체중 단독
    np.ones(len(train))
])

coeffs_full, _, _, _ = lstsq(X_full, train['Calories_Burned'].values, rcond=None)
pred_full = X_full @ coeffs_full
res_full = train['Calories_Burned'].values - pred_full

print(f"전체 곱셈 모델 RMSE: {rmse(train['Calories_Burned'].values, pred_full):.4f}")
print(f"잔차 std: {res_full.std():.4f}")

# 잔차 분석: 성별별 차이
male_mask = train['Gender'] == 'M'
print(f"\n남성 잔차 std: {res_full[male_mask].std():.4f}, mean: {res_full[male_mask].mean():.4f}")
print(f"여성 잔차 std: {res_full[~male_mask].std():.4f}, mean: {res_full[~male_mask].mean():.4f}")

### 분석 결과
1. **BPM, Age, Weight, Temp_diff 잔차 상관이 전부 0에 수렴** → 곱셈 구조가 데이터를 거의 완벽히 설명
2. **잔차 std ~6이 남음** → 설명 불가능한 노이즈 (데이터 내재)
3. **성별별 잔차 mean이 약간 다름** → 남/여 관계식이 다르다는 증거
4. → **성별 분리 모델의 이론적 근거 확보!**

---
## 6. 실험 — 성별 분리 모델

잔차 분석에서 발견한 "남/여 관계식 차이"를 반영
- M, F 각각 별도 모델 학습 → 예측 결합
- 핵심 5변수만 사용: Exercise_Duration, BPM, Temp_diff, Age, Weight(lb)
- Polynomial + Ridge + 정수 반올림

In [ ]:
# 성별 분리를 위한 데이터 준비
def prep_df(df):
    d = df.copy()
    d['Temp_diff'] = d['Body_Temperature(F)'] - 98.6
    return d

train_prep = prep_df(train)
test_prep = prep_df(test)

# 핵심 5변수 (파생변수 폭발 없이 Polynomial이 곱셈 구조를 학습)
CORE_COLS = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

# 성별 마스크
is_m_tr = (train_prep['Gender'] == 'M').values
is_f_tr = ~is_m_tr
is_m_te = (test_prep['Gender'] == 'M').values
is_f_te = ~is_m_te

y_all = train_prep['Calories_Burned'].values.astype(float)

print(f"남성 train: {is_m_tr.sum()}, 여성 train: {is_f_tr.sum()}")
print(f"남성 test: {is_m_te.sum()}, 여성 test: {is_f_te.sum()}")

In [ ]:
# 성별별 Poly+Ridge+Round 학습
from sklearn.linear_model import RidgeCV

for gender, mask in [('M', is_m_tr), ('F', is_f_tr)]:
    Xg = train_prep.loc[mask, CORE_COLS].values.astype(float)
    yg = y_all[mask]
    
    # OOF로 RMSE 확인
    oof = np.zeros(len(yg))
    for tr_idx, va_idx in kf.split(Xg):
        poly = PolynomialFeatures(degree=3, include_bias=False)
        sc = StandardScaler()
        X_tr = sc.fit_transform(poly.fit_transform(Xg[tr_idx]))
        X_va = sc.transform(poly.transform(Xg[va_idx]))
        
        model = Ridge(alpha=0.05, random_state=42)
        model.fit(X_tr, yg[tr_idx])
        oof[va_idx] = model.predict(X_va)
    
    oof_round = np.rint(np.clip(oof, 0, None))
    print(f"[{gender}] OOF RMSE (raw): {rmse(yg, oof):.6f}, (round): {rmse(yg, oof_round):.6f}")

---
## 결론

| 시도 | RMSE | 효과 |
|------|------|------|
| identity/sqrt 결합 | 0.299 | 스태킹 간소화 |
| + 정수 반올림 | 0.221 | 타겟 특성 반영 |
| + 잔차 분석 | - | 곱셈 구조 확인 |
| Degree 4 / MLP | 미미 | 대안 모델 효과 없음 |
| LightGBM+Keyser | 미미 | 물리식 추가 효과 없음 |
| **성별 분리** | **0.157** | **핵심 돌파** |

**핵심 인사이트**:
1. 정수 타겟 → round 적용만으로도 큰 개선
2. 잔차 분석이 성별 분리의 이론적 근거를 제공
3. 피처를 늘리는 것(MLP, Keyser)보다 데이터 구조에 맞는 분리가 더 효과적
4. → 최종 모델(05)에서 성별별 degree/alpha 그리드 탐색으로 발전